In [1]:
homedir = '/mnt/mirabelle/az6922_homedir/DRing/src/emp/datacentre/'
import random
import numpy as np

# from makec2s.ipynb
def genflowbytes():
    np.random.seed(0)
    
    mean_bytes = 100.0 * 1024
    shape = 1.05
    scale = mean_bytes * (shape - 1)/shape

    x = np.random.exponential(scale=1.0/shape)
    flowbytes = int(scale * np.exp(x))
    return flowbytes

def adjustbytesbymtu(flowbytes):
  mss = 1500
  return mss * ((flowbytes+mss-1)//mss)

large_flow_threshold = 10 * 1024 * 1024

In [2]:
stime = 144 # ms
bw = 1342176000 # B per second
seed_list = [1,2,3,4,5]
topologytype = 2
os = 1
nintervals = 8
cm = "prv1"
load = 2
swlist = range(140,201,20)
degree = 96
dringswpersn = 10

In [4]:
with open("rrg_copy_topologyfiles.conf", "w") as f:
    for sw in swlist:
        fromfile = f"{homedir}scalegraphfiles2/rrg_deg{degree}_sw{sw}_svr{sw*56}_os1_i1.edgelist"
        tofile = f"{homedir}evalscaletopologyfiles/rrg_{sw}_{degree}.edgelist"
        f.write(f"cp {fromfile} {tofile}\n")

actually copy

In [6]:
with open("rrg_copy_serverfiles.conf",'w') as f:
    for isw,sw in enumerate(swlist):
        fromfile = f"{homedir}serverfiles/rrg_{sw*56}_{sw}_{degree}"
        tofile = f"{homedir}evalserverfiles/rrg_{sw*56}_{sw}_{degree}.sv"
        f.write(f"cp {fromfile} {tofile}\n")

actually copy

In [7]:
with open("rrgsu2_generate_netpathfiles.conf",'w') as f:
    for isw,sw in enumerate(swlist):
        graphfile = f"{homedir}evalscaletopologyfiles/rrg_{sw}_{degree}.edgelist"
        npfile = f"{homedir}evalscalenetpathfiles/netpath_rrg_{sw}_{degree}_su2.np"
        f.write(f"python3 generate_suk_netpathfiles.py --graphfile {graphfile} --netpathfile {npfile} --numsw {sw} --korn 2\n")

(current dir: ~/DRing/src/emp/datacenter/)
python3 pararun.py --conf experiments/nsdi26fall/eval_scale_larger_ring/rrgsu2_generate_netpathfiles.conf --worker 10

In [8]:
for isw,sw in enumerate(swlist):
    # set parameters
    topologyfile = f"evalscaletopologyfiles/rrg_{sw}_{degree}.edgelist"
    serverfile = f"evalserverfiles/rrg_{sw*56}_{sw}_{degree}.sv"
    npfile = f"evalscalenetpathfiles/netpath_rrg_{sw}_{degree}_su2.np"
    nhosts = sw*56
    nswitches = sw
    k = degree
    with open(f"{homedir}{topologyfile}", 'r') as f:
        nlines = sum(1 for _ in f)
    nlinks = nlines * 2
    
    
    


    # generate connection_matrices file (1)
    unv1bytes = 0
    unv1file = f'{homedir}rawtrafficfiles/{cm}'
    maxinterval = 0
    with open(unv1file, 'r') as f:
        lines = f.readlines()
        for line in lines:
            tokens = line.split(',')
            # 0,32,31,10500
            # interval,fromserver,toserver,bytes
            unv1bytes += int(tokens[3])
            maxinterval = max(maxinterval, int(tokens[0]))
    print(f'unv1bytes {unv1bytes}, maxinterval {maxinterval}, fullload {bw * stime * nlinks / 1000}, ratio {(bw * stime * nlinks / 1000) / unv1bytes}')






    # generate connection_matrices file (2)
    random.seed(0)
    totalbytes = bw * stime / 1000 * nlinks * load / 100  # B
    mult = totalbytes / unv1bytes
    actualbytes = 0
    cmfile = f'cmfiles/rrg_sw{sw}_load{load}.cm'
    with open(cmfile, 'w') as fw:
        with open(unv1file, 'r') as fr:
            lines = fr.readlines()
            iline = 0
            while actualbytes < totalbytes:
                line = lines[iline]
                tokens = line.split(',')
                interval = int(tokens[0])
                fromserver = int(tokens[1])
                toserver = int(tokens[2])
                multbytes = int(tokens[3])

                if fromserver >= nhosts or toserver >= nhosts:
                    iline += 1
                    if iline >= len(lines):
                        iline = 0
                        if mult-1>0:
                            mult = mult-1
                    continue

                if mult >= 1 or (random.random() < mult):
                    multbytes = adjustbytesbymtu(multbytes)
    
                    # generate random start time
                    start_time_ms = random.uniform(0, stime//(maxinterval+1)) + interval * (stime//(maxinterval+1))

                    fw.write(f'{fromserver},{toserver},{int(multbytes)},{start_time_ms:.4f}\n')
                    actualbytes += int(multbytes)

                iline += 1
                if iline >= len(lines):
                    iline = 0
                    if mult-1>0:
                        mult = mult-1

                    # print(f'actualbytes {actualbytes}, totalbytes {totalbytes}, mult {mult}', end='\r')

    print(f'load {load}%, totalbytes {totalbytes}, unv1bytes {unv1bytes}, mult {mult}, actualbytes {actualbytes}')

unv1bytes 222240244500, maxinterval 7, fullload 1082330726400.0, ratio 4.870093303015602
load 2%, totalbytes 21646614528.0, unv1bytes 222240244500, mult 0.09740186606031204, actualbytes 21649390500
unv1bytes 222240244500, maxinterval 7, fullload 1236949401600.0, ratio 5.5658209177321165
load 2%, totalbytes 24738988032.0, unv1bytes 222240244500, mult 0.11131641835464233, actualbytes 24738990000
unv1bytes 222240244500, maxinterval 7, fullload 1391568076800.0, ratio 6.261548532448631
load 2%, totalbytes 27831361536.0, unv1bytes 222240244500, mult 0.12523097064897262, actualbytes 27831366000
unv1bytes 222240244500, maxinterval 7, fullload 1546186752000.0, ratio 6.957276147165146
load 2%, totalbytes 30923735040.0, unv1bytes 222240244500, mult 0.13914552294330293, actualbytes 30923791500


In [9]:
with open('rrgsu2_generate_pwfiles.conf', 'w') as fgen:
    with open('rrgsu2_copy_pwfiles.conf', 'w') as fcopy:
        for isw,sw in enumerate(swlist):
            # set parameters
            topologyfile = f"evalscaletopologyfiles/rrg_{sw}_{degree}.edgelist"
            serverfile = f"evalserverfiles/rrg_{sw*56}_{sw}_{degree}.sv"
            npfile = f"evalscalenetpathfiles/netpath_rrg_{sw}_{degree}_su2.np"
            nhosts = sw*56
            nswitches = sw
            k = degree
            with open(f"{homedir}{topologyfile}", 'r') as f:
                nlines = sum(1 for _ in f)
            nlinks = nlines * 2





            # generate pathweight file (1)
            interval_stime = stime / nintervals
            cmfile = f'cmfiles/rrg_sw{sw}_load{load}.cm'
            for interval in range(nintervals):
                flowstart = interval_stime * interval
                flowend = interval_stime * (interval + 1)
                varfile = f'{homedir}rawpathweightfiles/pathtraffic_rrg_sw{sw}_{nhosts}_{nswitches}_{k}_su2_prv1_load{load}_interval{interval}.var'
                qvarfile = f'{homedir}rawpathweightfiles/pathweight_rrg_sw{sw}_{nhosts}_{nswitches}_{k}_su2_prv1_load{load}_interval{interval}.var'
                fgen.write(f"python3 {homedir}generate_pathweightfiles.py --graphfile {homedir}{topologyfile} --serverfile {homedir}{serverfile} --numsw {nswitches} --numserver {nhosts} --netpathfile {homedir}{npfile} --flowfile {cmfile} --flowstart {flowstart} --flowend {flowend} --numfaillink 0 --linkfailurefile none --varfile {varfile} --qvarfile {qvarfile}\n")





            # generate pathweight file (2)
            intervaldict = {0:0,1:0,2:1,3:2,4:3,5:4,6:5,7:6} # to:from
            for interval in range(nintervals):
                fromfile = f'{homedir}rawpathweightfiles/pathweight_rrg_sw{sw}_{nhosts}_{nswitches}_{k}_su2_prv1_load{load}_interval{intervaldict[interval]}.var'
                tofile = f'{homedir}experiments/nsdi26fall/eval_scale_larger_ring/pwfiles/pathweight_rrg_sw{sw}_su2_prv1_load{load}_interval{interval}.pw'
                fcopy.write(f'cp {fromfile} {tofile}\n')

(current dir: ~/DRing/src/emp/datacentre/experiments/nsdi26fall/eval_scale_larger_ring/)
python3 ../../../pararun.py --conf rrgsu2_generate_pwfiles.conf --worker 40
+ actually copy rrgsu2_copy_pwfiles.conf

In [10]:
# generate conf file
conffile = f'{homedir}experiments/nsdi26fall/eval_scale_larger_ring/run_rrgsu2.conf'
with open(conffile, 'w') as frun:
    for isw,sw in enumerate(swlist):
        # set parameters
        topologyfile = f"evalscaletopologyfiles/rrg_{sw}_{degree}.edgelist"
        serverfile = f"evalserverfiles/rrg_{sw*56}_{sw}_{degree}.sv"
        npfile = f"evalscalenetpathfiles/netpath_rrg_{sw}_{degree}_su2.np"
        nhosts = sw*56
        nswitches = sw
        k = degree
        with open(f"{homedir}{topologyfile}", 'r') as f:
            nlines = sum(1 for _ in f)
        nlinks = nlines * 2





    
        for seed in seed_list:
            cmfile = f'experiments/nsdi26fall/eval_scale_larger_ring/cmfiles/rrg_sw{sw}_load{load}.cm'
            pwfileprefix = f'experiments/nsdi26fall/eval_scale_larger_ring/pwfiles/pathweight_rrg_sw{sw}_su2_prv1_load{load}_interval'
            outfile = f'experiments/nsdi26fall/eval_scale_larger_ring/outfiles/rrgsu2_sw{sw}_seed{seed}.out'
            frun.write(f"./eval -stime {stime} -seed {seed} -cmfile {cmfile} -topologytype {topologytype} -numswitches {nswitches} -numhosts {nhosts} -os {os} -ls_k {k} -npfile {npfile} -pwfileprefix {pwfileprefix} -numintervals {nintervals} -serverfile {serverfile} -topologyfile {topologyfile} > {outfile}\n")
            

python3 pararun.py --conf experiments/nsdi26fall/eval_scale_larger_ring/run_rrgsu2.conf --worker 10